# Suite LOG — The logical layout

`Logical` defines a diagram's logical layout: points, vectors, rotations and shapes, with no coordinates, and resolves the compass of a point or a shape: mbse-basis [CAP-237: logical layout schemas](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-237), by the rules of [the diagrams proposal](https://github.com/pitaman71/mbse-basis/blob/main/proposals/DIAGRAMS.md).

In [ ]:
from mbse.Schemas.Framework import Builders, Validators
from mbse.Diagrams.Framework import Logical

NAMES = ["Logical.OfRotation", "Logical.OfPoint", "Logical.Vector", "Logical.OfShape", "Logical.Origin",
         "Logical.Borders", "Logical.Extents", "Logical.Containment", "Logical.Adjacency"]
store = Builders.Store()
store.transaction(Logical.install)
L = store.Logical
problems = Validators.Validate(store).Reachable


def point():
    return L.OfPoint(lambda p: p)


def turn(degrees):
    return lambda r: r.degrees(degrees)


def raises(message, block):
    try:
        block()
    except ValueError as error:
        assert str(error) == message, str(error)
    else:
        raise AssertionError(f"expected ValueError: {message}")

<a id="LOG-01"></a>
## LOG-01 · The install defines the logical layout's schemas, each valid, and run again writes nothing

In [ ]:
schemas = [store.registered(name) for name in NAMES]
assert [schema.validate() for schema in schemas] == [[]] * len(NAMES)
store.transaction(Logical.install)
assert [store.registered(name) for name in NAMES] == schemas

<a id="LOG-02"></a>
## LOG-02 · A shape has one origin, and border and extent points of its own

In [ ]:
with store.transaction():
    o, b, e = point(), point(), point()
    shape = L.OfShape(lambda s: s
                      .origin(lambda a: a.insert(lambda x: x.point(o)))
                      .borders(lambda a: a.insert(lambda x: x.point(b)))
                      .extents(lambda a: a.insert(lambda x: x.point(e))))
    L.OfPoint(o, lambda p: p.outgoing(lambda a: a
                                      .insert(lambda x: x.end(b).rotation(turn(90.0)))
                                      .insert(lambda x: x.end(e).rotation(turn(180.0)))))
assert [x.point for x in shape.origin] == [o] and [x.point for x in shape.borders] == [b]
assert [x.point for x in shape.extents] == [e] and problems(L.OfShape, shape) == []

with store.transaction():
    L.OfShape(shape, lambda s: s.origin(lambda a: a.insert(lambda x: x.point(point()))))
assert problems(L.OfShape, shape) == ["unique(point) violated: entries agreeing on ['shape'] differ on ['point']"]
with store.transaction():
    L.OfShape(shape, lambda s: s.origin(lambda a: a.clear().insert(lambda x: x.point(o))))
    other = L.OfShape(lambda s: s.borders(lambda a: a.insert(lambda x: x.point(b))))
assert problems(L.OfShape, other) == ["unique(shape) violated: entries agreeing on ['point'] differ on ['shape']"]
with store.transaction():
    L.OfShape(other, lambda s: s.borders(lambda a: a.clear()))
assert problems(L.OfShape, shape) == []

<a id="LOG-03"></a>
## LOG-03 · A vector has one rotation per start and end

In [ ]:
with store.transaction():
    start, end = point(), point()
    L.OfPoint(start, lambda p: p.outgoing(lambda a: a.insert(lambda x: x.end(end).rotation(turn(45.0)))))
vector, = end.incoming
assert vector.start is start and vector.end is end and vector.rotation.degrees.get() == 45.0
assert problems(L.OfPoint, start) == []
with store.transaction():
    L.OfPoint(start, lambda p: p.outgoing(lambda a: a.insert(lambda x: x.end(end).rotation(turn(135.0)))))
assert problems(L.OfPoint, start) == [
    "unique(rotation) violated: entries agreeing on ['end', 'start'] differ on ['rotation']"]

<a id="LOG-04"></a>
## LOG-04 · A shape has one container, and adjacencies to its siblings by rotation

In [ ]:
with store.transaction():
    box, a, c = L.OfShape(lambda s: s), L.OfShape(lambda s: s), L.OfShape(lambda s: s)
    for child in (a, c):
        L.OfShape(child, lambda s: s.container(lambda x: x.insert(lambda e: e.container(box))))
    L.OfShape(a, lambda s: s.adjacent_to(lambda x: x.insert(lambda e: e.to_shape(c).rotation(turn(90.0)))))
assert [e.child for e in box.children] == [a, c] and problems(L.OfShape, box) == []
adjacency, = c.adjacent_from
assert adjacency.from_shape is a and adjacency.rotation.degrees.get() == 90.0
with store.transaction():
    L.OfShape(c, lambda s: s.container(lambda x: x.insert(lambda e: e.container(a))))
assert problems(L.OfShape, c) == ["unique(container) violated: entries agreeing on ['child'] differ on ['container']"]

<a id="LOG-05"></a>
## LOG-05 · A compass: a shape's and its origin's north, any other point's its preceding vector's start's, turned

In [ ]:
with store.transaction():
    p, q, r = point(), point(), point()
    L.OfPoint(b, lambda x: x.outgoing(lambda a: a.insert(lambda v: v.end(p).rotation(turn(300.0)))))
    L.OfPoint(p, lambda x: x.outgoing(lambda a: a.insert(lambda v: v.end(q).rotation(turn(-90.0)))))
    L.OfPoint(e, lambda x: x.outgoing(lambda a: a.insert(lambda v: v.end(r).rotation(turn(0.5)))))
assert (Logical.compass(shape), Logical.compass(o)) == (0.0, 0.0)
assert (Logical.compass(b), Logical.compass(e)) == (90.0, 180.0)
assert (Logical.compass(p), Logical.compass(q), Logical.compass(r)) == (30.0, 300.0, 180.5)

<a id="LOG-06"></a>
## LOG-06 · A compass the rules leave undefined is refused

In [ ]:
with store.transaction():
    lone, s, t, u, free = point(), point(), point(), point(), point()
    empty = L.OfShape(lambda x: x)
    L.OfPoint(lone, lambda x: x.outgoing(lambda a: a.insert(lambda v: v.end(s)).insert(lambda v: v.end(t).rotation(turn(1.0)))))
    L.OfPoint(o, lambda x: x.outgoing(lambda a: a.insert(lambda v: v.end(t).rotation(turn(2.0)))))
    L.OfPoint(t, lambda x: x.outgoing(lambda a: a.insert(lambda v: v.end(u).rotation(turn(3.0)))))
    L.OfPoint(u, lambda x: x.outgoing(lambda a: a.insert(lambda v: v.end(lone).rotation(turn(4.0)))))
raises("a shape without an origin has no compass", lambda: Logical.compass(empty))
raises("a point with 0 preceding vectors, not one, has no compass", lambda: Logical.compass(free))
raises("a point with 2 preceding vectors, not one, has no compass", lambda: Logical.compass(t))
raises("a point whose preceding vector has no rotation has no compass", lambda: Logical.compass(s))
with store.transaction():
    m, n = point(), point()
    L.OfPoint(m, lambda x: x.outgoing(lambda a: a.insert(lambda v: v.end(n).rotation(turn(5.0)))))
    L.OfPoint(n, lambda x: x.outgoing(lambda a: a.insert(lambda v: v.end(m).rotation(turn(6.0)))))
raises("a point on a cycle of vectors has no compass", lambda: Logical.compass(m))